# TP 1: LDA/QDA y optimización matemática de modelos

Se puede consultar la introducción teórica en el mono-notebook, se prefiere mantener este lo más chico posible.

In [ ]:
# imports
import numpy as np
import numpy.linalg as LA

from base.qda import QDA, TensorizedQDA, FasterQDA, EfficientQDA
from base.cholesky import QDA_Chol1, QDA_Chol2, QDA_Chol3
from utils.bench import Benchmark
from utils.datasets import (get_iris_dataset, get_letters_dataset, 
                            get_penguins_dataset, get_wine_dataset,
                            label_encode)


SyntaxError: trailing comma not allowed without surrounding parentheses (2653444046.py, line 8)

## Comparación benchmarks

In [ ]:
X_full, y_full = get_penguins_dataset()

X_full.shape, y_full.shape

((342, 4), (342, 1))

In [21]:
# encodeamos a número las clases
y_full_encoded = label_encode(y_full)

y_full[:5], y_full_encoded[:5]

(array([['Adelie'],
        ['Adelie'],
        ['Adelie'],
        ['Adelie'],
        ['Adelie']], dtype=object),
 array([[0],
        [0],
        [0],
        [0],
        [0]]))

In [25]:
# generamos el benchmark
# observar que son valores muy bajos de runs para que corra rápido ahora
b = Benchmark(
    X_full, y_full_encoded,
    n_runs = 50,
    warmup = 10,
    mem_runs = 5,
    test_sz = 0.3,
    same_splits = False
)

Benching params:
Total runs: 65
Warmup runs: 10
Peak Memory usage runs: 5
Running time runs: 50
Train size rows (approx): 240
Test size rows (approx): 102
Test size fraction: 0.3


In [26]:
# bencheamos un par
to_bench = [QDA, TensorizedQDA, FasterQDA, EfficientQDA]

for model in to_bench:
    b.bench(model)

QDA (MEM):   0%|          | 0/5 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/50 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/5 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/50 [00:00<?, ?it/s]

FasterQDA (MEM):   0%|          | 0/5 [00:00<?, ?it/s]

FasterQDA (TIME):   0%|          | 0/50 [00:00<?, ?it/s]

EfficientQDA (MEM):   0%|          | 0/5 [00:00<?, ?it/s]

EfficientQDA (TIME):   0%|          | 0/50 [00:00<?, ?it/s]

In [51]:
# hacemos un summary
summ = b.summary()
summ[['train_median_ms', 'test_median_ms', 'train_mem_median_mb', 'test_mem_median_mb']]

,train_median_ms,test_median_ms,train_mem_median_mb,test_mem_median_mb
model,,,,
QDA,0.244341,3.167082,0.011147,0.004423
TensorizedQDA,0.213367,1.078532,0.011421,0.004669
FasterQDA,0.219443,0.062334,0.010834,0.264503
EfficientQDA,0.218326,0.044384,0.011192,0.041016


## Conclusiones (consigna #7)
- `train_median_ms`: es prácticamente el mismo para todos los modelos, lo cual está bien porque el código es el mismo.
- `test_median_ms`: esta sección es una de las más importantes, acá sí podemos apreciar cómo varía el tiempo de ejecución. Más específicamente, se puede apreciar cómo fuimos reduciendo considerablemente y, por ende, mejorando en cada modelo.
- `test_mem_median_mb`: esta sección también es de las más importantes, acá podemos apreciar cómo cambiar el consumo de memoria para cada modelo. A resaltar, FasterQDA consume una mucho mayor que sus pares: 56 veces más que TensorizedQDA. Esto es muy importante y a considerar para dataset de gran tamaño. En cuanto a EfficientQDA, podemos ver que consume más memoria que TensorizedQDA (8x), pero al mismo tiempo es considerablemente más rápido (24x)

En líneas generales, podemos observar cómo EfficientQDA ofrece una muy buena relación en cuanto a tiempo de procesamiento y memoria utilizada. Para el modelo de FasterQDA, que al eliminar el loop en la predicción y utilizar todo el dataset para el cálculo, logramos mejorar considerablemente el tiempo de ejecución, pero a costa de un consumo de memoria muy elevado.

Las observaciones condicen con lo esperado y sorprende el hecho de las grandes mejoras que se logra aplicando soluciones con un trasfondo matemático.